<a href="https://colab.research.google.com/github/Venkateshpaitwar/Code-Comment-Quality-Analyzer-NLP/blob/main/dashboard/comment_analysis_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
!pip install streamlit pyngrok textstat

In [33]:
!ngrok config add-authtoken AUTH_TOKEN

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [23]:
from pyngrok import ngrok

In [36]:
%%writefile app.py
import streamlit as st
import pandas as pd
import re
import tokenize
from io import StringIO
import textstat

# Page configuration
st.set_page_config(
    page_title="Code Comment Quality Analyzer",
    page_icon="💬",
    layout="wide"
)

st.title("💬 Code Comment Quality Analyzer")

st.markdown(
"""
Upload a **Python (.py) file** to analyze the **quality and type of comments** using NLP techniques.

The system will:
- Extract comments
- Classify comment type
- Compute comment quality score
- Display visual analytics
"""
)

uploaded_file = st.file_uploader("Upload a Python file", type=["py"])


def extract_comments(code):
    comments = []
    tokens = tokenize.generate_tokens(StringIO(code).readline)

    for toknum, tokval, _, _, _ in tokens:
        if toknum == tokenize.COMMENT:
            comments.append(tokval)

    return comments


def clean_comment(text):
    text = text.lower()
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)
    return text.strip()


def classify_comment(text):

    if "todo" in text or "fixme" in text:
        return "TODO"

    elif "debug" in text or "print" in text:
        return "DEBUG"

    elif "return" in text or "param" in text:
        return "DOCSTRING"

    else:
        return "EXPLANATION"


def quality_score(text):

    length_score = min(len(text.split()) / 10, 1)

    readability_score = textstat.flesch_reading_ease(text) / 100
    readability_score = max(0, min(readability_score, 1))

    score = (length_score * 0.5) + (readability_score * 0.5)

    return round(score, 2)


if uploaded_file is not None:

    code = uploaded_file.read().decode("utf-8")

    st.subheader("📄 Uploaded Code")
    st.code(code, language="python")

    comments = extract_comments(code)

    cleaned = [clean_comment(c) for c in comments]

    data = []

    for comment in cleaned:

        label = classify_comment(comment)
        score = quality_score(comment)

        data.append({
            "comment": comment,
            "type": label,
            "quality_score": score
        })

    df = pd.DataFrame(data)

    # Metrics
    st.subheader("📊 Overview")

    col1, col2, col3 = st.columns(3)

    col1.metric("Total Comments", len(df))
    col2.metric("Average Quality Score", round(df["quality_score"].mean(), 2))
    col3.metric("Comment Types", df["type"].nunique())

    # Quality summary
    st.subheader("📦 Quality Summary")

    high = len(df[df["quality_score"] > 0.7])
    medium = len(df[(df["quality_score"] >= 0.4) & (df["quality_score"] <= 0.7)])
    low = len(df[df["quality_score"] < 0.4])

    col1, col2, col3 = st.columns(3)

    col1.metric("High Quality", high)
    col2.metric("Medium Quality", medium)
    col3.metric("Low Quality", low)

    # Comment type distribution
    st.subheader("📊 Comment Type Distribution")

    type_counts = df["type"].value_counts()

    st.bar_chart(type_counts)

    # Quality distribution
    st.subheader("📈 Quality Score Distribution")

    st.bar_chart(df["quality_score"])

    # Results table
    st.subheader("📋 Comment Analysis Results")

    st.dataframe(df, use_container_width=True)

Overwriting app.py


In [37]:
!streamlit run app.py &>/content/logs.txt &

In [38]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://libratory-johana-irrelative.ngrok-free.dev" -> "http://localhost:8501"


# Author : Venkatesh Paitwar